# Transformer world models — what is the state, and can you edit it?

**Direction:** `research/directions/transformer-world-state.md` · `[reframe]` · sub-Q 1, 2, 3.
**Branch:** `delta_h_analysis`. First pass.

## Why this notebook is different from every previous one

Every §4 result so far assumes the model has **one** state: a vector `h` that is both what the
model carries forward and what a probe reads. Writing to it *is* the intervention. A transformer
breaks that assumption, and the break is the point of this notebook.

A causal transformer has **two** state objects, and they come apart:

| | what it is | carried? | history-dependent? |
|---|---|---|---|
| **carried state** — the observation buffer | the recent frames the model must be given to reproduce its own prediction | **yes** | each element is a function of *one* frame |
| **readable state** — residual stream at (layer ℓ, current position) | what attention has mixed together at this position | **no** — recomputed every step | **yes** |

In a GRU these are the same object. Here, the thing you can *read* the world out of is not the
thing that *persists*, so "edit the world state" splits into two different experiments — and a
write to the readable state is transient **by construction**, not by failure.

## The carried state is bigger than the attention window

Stacking layers widens the receptive field: at layer L, position `t` depends on positions
`t − L·(W−1) … t`, because each layer's keys were themselves computed from a window. So the
carried state spans

$$\texttt{state\_span} \;=\; \texttt{n\_layers}\times(\texttt{window}-1)+1$$

frames, and `window` is only the *per-layer attention span*. This is verified numerically in
`tests/test_transformer.py`: a one-pass banded-mask forward and a step-by-step buffer rollout
agree to float tolerance **only** when the buffer holds `state_span` frames, and diverge from
exactly `t = window` onward otherwise. Sizing the buffer by `window` would badly mis-state how
much history an edit has to overwrite — which is the headline experiment in §5.

## Definitions

### Runs (canonical registry: `TRANSFORMER_RUNS.md`)

All trained by `scripts/train_transformer.py` on `datasets/4_fixed_refl_inview` (obs noise 0.2,
position noise 0.04) — the same data as every other architecture. Same MSE next-step objective as
the GRU; `d_model = 256` **matches the GRU's hidden size** so state-geometry numbers are directly
comparable (row-space chance level is `√(d/H)`, so the width must match). 300 epochs, batch 256,
AdamW lr 1e-3 with 5% warmup + cosine decay, grad-clip 1.0.

| code | descriptive label (used in every figure) | window | carried `state_span` | note |
|---|---|---|---|---|
| `W2` | **transformer · window 2** | 2 | 5 frames | the minimum that can see velocity |
| `W4` | **transformer · window 4** | 4 | 13 frames | |
| `W16` | **transformer · window 16** | 16 | 61 frames | exceeds the 20 frames available before `ef`, so its *effective* carried state is the whole history — full context |
| `H256` | **GRU · H=256 (reference)** | — | unbounded (compressed into 256 dims) | from `../controls/CONTROL_RUNS.md` |

**`window` is an explanatory variable, not a robustness check.** It dials continuously between
"no compressed state, just a lookup over raw history" (large W) and "history must be compressed
into the residual stream" (small W) — so it is the *mechanism* behind any GRU-vs-transformer
difference rather than a sensitivity analysis of it.

### Residual points (the layer axis)

`probe_layer` indexes **residual points**, of which there are `n_layers + 1 = 5`:

| point | what it is |
|---|---|
| **0** | the encoder output `relu(Linear(obs))` — the **encoder port**, identical in form to the GRU's `x` |
| 1, 2, 3 | block inputs — labelled **early / middle** below |
| **4** | the final pre-LayerNorm stream the decoder reads — **last** |

An edit at residual point ℓ changes the block inputs for layers **> ℓ only**. So editing the *last*
point alters this position's own prediction and propagates to nothing, while editing point 0
propagates furthest. Readability and writability need not peak at the same depth, which is exactly
what §4 measures.

### Metrics

The canonical §4 set (`../METRICS_AND_EDITORS.md` §4, implemented in `scripts/editability_metrics.py`
— imported, never re-derived): **Edit Index** ∈ [−1,+1] (+1 = the output *is* the world where the
edit happened, −1 = the world where it did not, 0 = equidistant from both), **Edit Index by step**,
**Target / Ghost / Collateral / Edit-frame RMSE**, **GT-traj RMSE**, **fidelity ratio**.
Plus §1 geometry (PCA hull, TwoNN / MLE intrinsic dimension), §2 recoverability (position and
velocity R², linear and MLP, held out), §3 canonicality (fiber residual).

Two metrics are **introduced here** because the history-overwrite sweep needs them (both folded back
into the registry):

| metric | formula | units | better | what it reads |
|---|---|---|---|---|
| **crossover point** | smallest `n` with Edit Index > 0 | frames | ↓ | the output stops being closer to the *unedited* world. A **low bar** — a single frame can clear it — so it does not discriminate between models. |
| **saturation point** | smallest `n` with Edit Index ≥ `0.9 × max_n(Edit Index)` **for that same model** | frames (also shown as % of span) | ↓ | the point past which more overwritten history buys nothing. This is the number that adjudicates the two registered predictions in §5. |

**Spans, and which one the percentages use.** `state_span` is the *architectural* carried state. But at
edit frame `ef = 20` the model has only ever seen 20 frames, so a run whose `state_span` exceeds `ef`
has a shorter **effective** carried state. The two differ only for `W16` (architectural 61, effective 20).
Percentages are reported against the **effective** span — the history that actually exists — with the
architectural column shown alongside so the difference is visible rather than hidden.

> **Read the Edit Index against each model's own unsteered row.** A perfect predictor scores exactly
> −1 unsteered; a real one falls short by its own blur, so the −1 end sits at a slightly different
> place per model. Every table below reports it.

### Interventions

| name | acts on | persists? |
|---|---|---|
| **Activation edit (layer ℓ)** | the *readable* state — residual stream at ℓ, current position | **no** by construction: the next step recomputes from the buffer |
| **History overwrite (n frames)** | the *carried* state — the last n frames of the observation buffer, replaced by renders of the counterfactual world | **yes** — this is the only channel that persists |
| Oracle observation | one extra teacher-forced frame: the real (noisy) post-edit observation | reference |
| Unsteered / GT (sim) | references, never editors | — |

In [ ]:
# [1] Setup: load the window sweep + the GRU reference, the edits split, and check the ±1 alignment.
import os, sys, json, time
sys.path.insert(0, "../../../..")
sys.path.insert(0, "../../../../scripts")
import numpy as np, torch, h5py
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from IPython.display import display, Markdown

from pim.world_models import load_checkpoint, load_dataset
from pim.simulator.sim import Scene, SimConfig
from pim.simulator.renderer import render_scene
from pim.figures.theme import style_ax
from editability_metrics import (build_edit_zones, edit_scorecard, edit_index_by_step,
                                 fidelity_ratio)

torch.manual_seed(0); np.random.seed(0)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
N_OBJ, K_ROLL, N_EVAL, N_PROBE = 2, 15, 192, 1200
OUT = "/tmp/transformer_wm"; os.makedirs(OUT, exist_ok=True)

TF_RUNS = ["W2", "W4", "W16"]
LABEL = {"W2": "transformer · window 2", "W4": "transformer · window 4",
         "W16": "transformer · window 16", "H256": "GRU · H=256 (reference)"}
COLOR = {"W2": "#0072B2", "W4": "#009E73", "W16": "#CC79A7", "H256": "0.45"}

MODELS = {}
for r in TF_RUNS:
    MODELS[r], _ = load_checkpoint(f"../../../../runs/transformers/{r}/best_model.pt", device=DEVICE)
MODELS["H256"], _ = load_checkpoint("../../../../runs/controls/H256/best_model.pt", device=DEVICE)
ALL = list(MODELS)
IS_TF = {k: hasattr(m, "state_span") for k, m in MODELS.items()}
N_LAYERS = MODELS["W16"].cfg.n_layers
LAYER_PTS = list(range(N_LAYERS + 1))
LAYER_NAME = {0: "0 · encoder port", 1: "1 · early", 2: "2 · middle",
              N_LAYERS - 1: f"{N_LAYERS-1} · late", N_LAYERS: f"{N_LAYERS} · last (decoder input)"}
LAYER_TICK = {0: "0\nencoder\nport", 1: "1\nearly", 2: "2\nmiddle",
              N_LAYERS - 1: f"{N_LAYERS-1}\nlate", N_LAYERS: f"{N_LAYERS}\nlast"}
LAYER_TRIO = [0, N_LAYERS // 2, N_LAYERS]      # encoder port / middle / last

def top_legend(fig, handles, ncol=None, y=0.995):
    """One shared legend across the top of the figure — never inside a panel, so that adding a
    4th/5th model widens the figure instead of crowding a panel (CLAUDE.md, N-model rule)."""
    fig.legend(handles=handles, loc="upper center", bbox_to_anchor=(0.5, y),
               ncol=ncol or min(len(handles), 5), fontsize=8.5, frameon=False)

def mline(r, **kw):
    """Legend swatch for a model. The GRU is always drawn as a marker-less dashed reference line,
    so its swatch must be too — a swatch that shows a marker the plot never draws is a lie."""
    kw.setdefault("color", COLOR[r]); kw.setdefault("lw", 1.6)
    if r == "H256":
        kw.pop("marker", None); kw.pop("ms", None)
    return Line2D([], [], ls="--" if r == "H256" else "-", label=LABEL[r], **kw)

bundle = load_dataset("../../../../datasets/4_fixed_refl_inview", n_obj_keep=N_OBJ)
edits, test = bundle.edits, bundle.test
ef = edits.edit_frame; sim = test.config["dataset"]["sim"]; R = edits.obs_res
with h5py.File(edits.h5_path, "r") as f:
    VEL_E = f["velocities"][:, :, :N_OBJ, :].astype(np.float32)

@torch.no_grad()
def warm(model, obs_np, upto):
    """Flat state aligned so decode(state) predicts sim frame `upto` (GRU convention)."""
    o = torch.from_numpy(obs_np).float().to(DEVICE)
    if hasattr(model, "state_from_obs"):
        return model.state_from_obs(o[:, :upto])
    st = None
    for t in range(upto):
        _, st = model.step(o[:, t], st)
    return st

@torch.no_grad()
def roll(model, state, steps=K_ROLL):
    out = [model.decode(state)]
    s = state
    for _ in range(steps - 1):
        p, s = model.predict_step(s); out.append(p)
    return torch.stack(out, 1).cpu().numpy()

def run_dir(r):
    return "../../../../runs/transformers" if IS_TF[r] else "../../../../runs/controls"
def best_val(r):
    return min(json.loads(x)["val_loss"] for x in open(f"{run_dir(r)}/{r}/metrics.jsonl"))
print(f"{'run':<6}{'params':>11}{'window':>8}{'span':>7}{'best val':>11}")
for r in ALL:
    m = MODELS[r]
    w = m.cfg.window if IS_TF[r] else "-"
    sp = m.state_span if IS_TF[r] else "-"
    print(f"{r:<6}{sum(q.numel() for q in m.parameters()):>11,}{str(w):>8}{str(sp):>7}{best_val(r):>11.5f}")

# alignment check on ORDINARY (non-edit) sequences — must NOT use the edits split,
# where the pre-edit state legitimately fails to predict frame ef (that IS the experiment)
print("\nalignment check (test split) — RMSE(decode(warm to t), clean_obs[t+k]); min must be at k=0:")
for r in ALL:
    m = MODELS[r]; st = warm(m, test.obs[:256].astype(np.float32), ef)
    d = m.decode(st).cpu().numpy()
    e = {k: float(np.sqrt(((d - test.clean_obs[:256, ef+k])**2).mean())) for k in (-1, 0, 1)}
    b = min(e, key=e.get)
    print(f"  {r:<5s} k=-1 {e[-1]:.4f} | k=0 {e[0]:.4f} | k=+1 {e[1]:.4f} -> min k={b} {'PASS' if b==0 else 'FAIL'}")

---
## §1 — Predictive quality (the gate)

*Sections here run in dependency order and are numbered sequentially. The mapping to
`00_master_editability` is: this §1 is its quality gate, §2 below covers its **§2 recoverability** and
**§3 canonicality**, §3 below is its **§1 geometry**, and §4–§5 are its **§4 editability**.*

Editability numbers are uninterpretable from a weak predictor, so this gates everything below.
The transformer must reach the GRU's next-step RMSE before any §4 claim is worth making.

In [ ]:
# [2] Fig 1 — predictive quality: training curves, free-run decay, and the quality gate.
from pim.eval.baselines import compute_obs_baselines
bl = compute_obs_baselines(test.obs[:1000], test.clean_obs[:1000], float(sim["obs_noise_std"]))

@torch.no_grad()
def next_step_rmse(model, n=1000, batch=250):
    se, cnt = 0.0, 0
    for i in range(0, n, batch):
        o = torch.from_numpy(test.obs[i:i+batch]).float().to(DEVICE)
        pr, _ = model(o)
        gt = test.clean_obs[i:i+batch, 1:]
        se += float(((pr.cpu().numpy() - gt)**2).sum()); cnt += gt.size
    return float(np.sqrt(se/cnt))

@torch.no_grad()
def freerun(model, steps=20, warm_to=10, n=800, batch=200):
    per, cnt = np.zeros(steps), 0
    for i in range(0, n, batch):
        st = warm(model, test.obs[i:i+batch].astype(np.float32), warm_to)
        r = roll(model, st, steps)
        gt = test.clean_obs[i:i+batch, warm_to:warm_to+steps]
        per += ((r-gt)**2).mean(axis=(0,2))*len(gt); cnt += len(gt)
    return np.sqrt(per/cnt)

PRED = {r: dict(next=next_step_rmse(MODELS[r]), fr=freerun(MODELS[r])) for r in ALL}
plt.style.use("default")
fig, ax = plt.subplots(1, 3, figsize=(17, 4.3))
BEST_EP = {}
for j, r in enumerate(ALL):
    hist = [json.loads(x) for x in open(f"{run_dir(r)}/{r}/metrics.jsonl")]
    ax[0].plot([h["epoch"] for h in hist], [h["val_loss"] for h in hist],
               color=COLOR[r], lw=1.5, ls="--" if r == "H256" else "-", label=LABEL[r])
    be = min(hist, key=lambda h: h["val_loss"]); BEST_EP[r] = be["epoch"]
    ax[0].plot(be["epoch"], be["val_loss"], "o", ms=6, mfc="none", mew=1.6, color=COLOR[r])
# The transformers all bottom out within a few epochs of each other, so per-point labels would
# overlap into an unreadable smudge — list the best epochs once, in a corner block instead.
ax[0].annotate("open circle = best epoch\n" + "\n".join(
                   f"{LABEL[r].replace(' (reference)', '')}: {BEST_EP[r]}" for r in ALL),
               xy=(0.97, 0.95), xycoords="axes fraction", fontsize=7, color="0.25",
               ha="right", va="top",
               bbox=dict(boxstyle="round,pad=0.35", fc="white", ec="0.75", lw=0.7, alpha=0.9))
ax[0].set_xlabel("epoch"); ax[0].set_ylabel("validation MSE"); ax[0].set_yscale("log")
ax[0].set_title("(a) training — open circle marks the checkpoint actually used", fontsize=9.5)
s = np.arange(len(PRED["W16"]["fr"]))
for r in ALL:
    ax[1].plot(s, PRED[r]["fr"], marker="o", ms=3, color=COLOR[r],
               ls="--" if r == "H256" else "-", label=LABEL[r])
BASE = [(bl.identity_rmse, "#8B4513", "copy previous frame"),
        (bl.noise_floor_rmse, "#D55E00", "observation noise floor"),
        (bl.random_rmse, "0.6", "random frame")]
for v, c, lb in BASE:
    ax[1].axhline(v, ls=":", lw=1.2, color=c)
    ax[1].annotate(f"{lb} ({v:.3f})", xy=(0.02, v), xycoords=("axes fraction", "data"),
                   fontsize=7, color=c, va="bottom")
ax[1].set_xlabel("free-run step"); ax[1].set_ylabel("RMSE vs clean observation")
ax[1].set_title("(b) free-run decay — no teacher forcing after step 0", fontsize=9.5)
xs = np.arange(len(ALL))
ax[2].bar(xs, [PRED[r]["next"] for r in ALL], 0.6, color=[COLOR[r] for r in ALL])
ax[2].axhline(PRED["H256"]["next"], color="0.3", ls="--", lw=1.2)
ax[2].annotate("GRU reference", xy=(len(ALL)-0.45, PRED["H256"]["next"]), fontsize=7.5,
               color="0.35", ha="right", va="bottom")
ax[2].axhline(bl.noise_floor_rmse, color="#D55E00", ls=":", lw=1.2)
ax[2].annotate(f"observation noise floor ({bl.noise_floor_rmse:.3f}) — nothing can beat this",
               xy=(-0.45, bl.noise_floor_rmse), fontsize=7.5, color="#D55E00", ha="left", va="bottom")
ax[2].set_ylim(0, bl.noise_floor_rmse*1.18)
ax[2].set_xticks(xs); ax[2].set_xticklabels([LABEL[r].replace(" · ", "\n") for r in ALL], fontsize=7.5)
ax[2].set_ylabel("next-step RMSE vs clean")
ax[2].set_title("(c) quality gate — all models must sit at the GRU's level", fontsize=9.5)
for a in ax: a.grid(alpha=0.3); style_ax(a)
top_legend(fig, [mline(r, marker="o", ms=4) for r in ALL])
fig.suptitle("Fig 1 — predictive quality: does the transformer match the GRU before we interpret any edit?",
             y=1.10, fontsize=12)
fig.tight_layout(rect=[0, 0, 1, 0.94])
fig.savefig(f"{OUT}/fig1_predictive.png", dpi=130, bbox_inches="tight")
display(fig); plt.close(fig)
for r in ALL:
    print(f"  {LABEL[r]:<34s} next-step RMSE {PRED[r]['next']:.4f}  "
          f"({PRED[r]['next']/PRED['H256']['next']:.2f}x the GRU)")

---
## §2 — Where is the world state readable? (recoverability and canonicality)

The transformer-specific question. The GRU has one place to probe; the transformer has one per
residual point, per window. Position and velocity R² and the fiber residual are computed at every
residual point, so "which layer is the world state" becomes a measurement rather than a choice.

**Fiber residual** (canonicality) is the fraction of ‖state‖ that is *not* a function of `(position,
velocity)` — high means the state carries a lot beyond the physical variables. It is reported for the
linear and MLP probe in Table 1 and plotted for the MLP probe in Fig 2c.

In [ ]:
# [3] Fig 2 — recoverability and canonicality as a function of depth (the transformer-specific view).
from eval_controls import recoverability_and_canonicality

obs_p = test.obs[:N_PROBE].astype(np.float32)
P_pos = test.positions[:N_PROBE, :, :N_OBJ, :].reshape(N_PROBE, -1, N_OBJ*2)
with h5py.File(test.h5_path, "r") as f:
    P_vel = f["velocities"][:N_PROBE, :, :N_OBJ, :].astype(np.float32).reshape(N_PROBE, -1, N_OBJ*2)
vis = test.is_visible[:N_PROBE, :, :N_OBJ].all(axis=2)

@torch.no_grad()
def states_at(model, layer=None, batch=300):
    out = []
    for i in range(0, len(obs_p), batch):
        o = torch.from_numpy(obs_p[i:i+batch]).float().to(DEVICE)
        if layer is not None:
            model.state_view = "activations"; model.probe_layer = layer
        out.append(model.get_hidden_states(o).cpu().numpy())
    return np.concatenate(out, 0)

DEPTH = {}
for r in TF_RUNS:
    DEPTH[r] = {}
    for L in LAYER_PTS:
        Hs = states_at(MODELS[r], L); T = Hs.shape[1]; m_ = vis[:, :T]
        DEPTH[r][L] = recoverability_and_canonicality(
            Hs[m_], P_pos[:, :T][m_], P_vel[:, :T][m_])
Hs_g = states_at(MODELS["H256"]); Tg = Hs_g.shape[1]; mg = vis[:, :Tg]
GRU_REC = recoverability_and_canonicality(Hs_g[mg], P_pos[:, :Tg][mg], P_vel[:, :Tg][mg])

plt.style.use("default")
fig, ax = plt.subplots(1, 3, figsize=(17, 4.3))
panels = [("pos_r2_linear", "position R² (linear probe)", "(a) is position readable?"),
          ("vel_r2_linear", "velocity R² (linear probe)", "(b) is velocity readable?"),
          ("fiber_resid_mlp", "fiber residual (MLP probe)",
           "(c) how much of the state is not physical state?")]
for i, (key, ylab, title) in enumerate(panels):
    for r in TF_RUNS:
        ax[i].plot(LAYER_PTS, [DEPTH[r][L][key] for L in LAYER_PTS], "-o", ms=5, color=COLOR[r])
    ax[i].axhline(GRU_REC[key], color=COLOR["H256"], ls="--", lw=1.4)
    ax[i].set_xticks(LAYER_PTS)
    ax[i].set_xticklabels([LAYER_TICK.get(L, str(L)) for L in LAYER_PTS], fontsize=8)
    ax[i].set_xlabel("residual point edited / probed"); ax[i].set_ylabel(ylab, fontsize=9.5)
    ax[i].set_title(title, fontsize=9.5)
    ax[i].grid(alpha=0.3); style_ax(ax[i])
top_legend(fig, [mline(r, marker="o", ms=4) for r in ALL])
fig.suptitle("Fig 2 — where in the network is the world state readable? "
             "The GRU has one state; the transformer has one per residual point.",
             y=1.10, fontsize=12)
fig.tight_layout(rect=[0, 0, 1, 0.94])
fig.savefig(f"{OUT}/fig2_depth_readability.png", dpi=130, bbox_inches="tight")
display(fig); plt.close(fig)

rows = ["| model | residual point | position R² lin / MLP | velocity R² lin / MLP | fiber resid lin / MLP |",
        "|---|---|---|---|---|"]
for r in TF_RUNS:
    for L in LAYER_TRIO:
        d = DEPTH[r][L]
        rows.append(f"| {LABEL[r]} | {LAYER_NAME.get(L, L)} | {d['pos_r2_linear']:.3f} / {d['pos_r2_mlp']:.3f} | "
                    f"{d['vel_r2_linear']:.3f} / {d['vel_r2_mlp']:.3f} | "
                    f"{d['fiber_resid_linear']:.3f} / {d['fiber_resid_mlp']:.3f} |")
rows.append(f"| {LABEL['H256']} | (single state) | {GRU_REC['pos_r2_linear']:.3f} / {GRU_REC['pos_r2_mlp']:.3f} | "
            f"{GRU_REC['vel_r2_linear']:.3f} / {GRU_REC['vel_r2_mlp']:.3f} | "
            f"{GRU_REC['fiber_resid_linear']:.3f} / {GRU_REC['fiber_resid_mlp']:.3f} |")
display(Markdown("**Table 1 — recoverability and canonicality by depth** (held-out 30%, frames where both "
                 "objects are visible)\n\n" + "\n".join(rows)))

---
## §3 — Geometry of the visited-state manifold

Same estimators as `00_master_editability` §1: the linear PCA hull and two model-free intrinsic-dimension
estimators. Physical reference: the world has **8 degrees of freedom** (2 objects × (x, y, vx, vy)).

In [ ]:
# [4] Fig 3 — geometry: PCA hull and intrinsic dimension of the visited states, per model.
def pca_spectrum(X, k=64):
    Xc = X - X.mean(0); C = (Xc.T @ Xc) / max(len(Xc)-1, 1)
    ev = np.linalg.eigvalsh(C)[::-1].clip(0)
    return ev / max(ev.sum(), 1e-12)

def twonn(X, n=3000, seed=0):
    g = np.random.default_rng(seed); Z = X[g.permutation(len(X))[:n]]
    D = np.linalg.norm(Z[:, None] - Z[None], axis=-1); np.fill_diagonal(D, np.inf)
    r = np.sort(D, axis=1)[:, :2]
    mu = r[:, 1] / np.maximum(r[:, 0], 1e-12); mu = mu[np.isfinite(mu) & (mu > 1)]
    return float(len(mu) / np.log(mu).sum())

def mle_dim(X, k=20, n=3000, seed=0):
    g = np.random.default_rng(seed); Z = X[g.permutation(len(X))[:n]]
    D = np.linalg.norm(Z[:, None] - Z[None], axis=-1); np.fill_diagonal(D, np.inf)
    r = np.sort(D, axis=1)[:, :k]
    inv = np.log(np.maximum(r[:, k-1:k], 1e-12)) - np.log(np.maximum(r[:, :k-1], 1e-12))
    d = (k - 2) / np.maximum(inv.mean(1) * (k - 1), 1e-12)
    return float(np.mean(d))

GEO = {}
for r in TF_RUNS:
    Hs = states_at(MODELS[r], N_LAYERS); X = Hs.reshape(-1, Hs.shape[-1])[:6000]
    GEO[r] = dict(spec=pca_spectrum(X), twonn=twonn(X), mle=mle_dim(X))
Xg = Hs_g.reshape(-1, Hs_g.shape[-1])[:6000]
GEO["H256"] = dict(spec=pca_spectrum(Xg), twonn=twonn(Xg), mle=mle_dim(Xg))

plt.style.use("default")
fig, ax = plt.subplots(1, 2, figsize=(13, 4.3))
for r in ALL:
    cum = np.cumsum(GEO[r]["spec"])
    ax[0].plot(np.arange(1, len(cum)+1), cum, color=COLOR[r], lw=1.8,
               ls="--" if r == "H256" else "-", label=LABEL[r])
for lv in (0.90, 0.99):
    ax[0].axhline(lv, color="0.6", ls=":", lw=1)
ax[0].axvline(8, color="#D55E00", ls=":", lw=1.4)
ax[0].annotate("8 = the world's degrees of freedom\n(2 objects × x, y, vx, vy)",
               xy=(8.6, 0.28), fontsize=8, color="#D55E00")
ax[0].set_xscale("log"); ax[0].set_xlabel("PCA components"); ax[0].set_ylabel("cumulative variance")
ax[0].set_title("(a) linear hull of the visited states", fontsize=9.5)
xs = np.arange(len(ALL)); w = 0.38
ax[1].bar(xs-w/2, [GEO[r]["twonn"] for r in ALL], w, color="#0072B2", label="TwoNN")
ax[1].bar(xs+w/2, [GEO[r]["mle"] for r in ALL], w, color="#E69F00", label="MLE (k=20)")
ax[1].axhline(8, color="#D55E00", ls=":", lw=1.4)
ax[1].annotate("8 = the world's degrees of freedom", xy=(len(ALL)-0.45, 8.15), fontsize=8,
               color="#D55E00", ha="right")
ax[1].set_xticks(xs); ax[1].set_xticklabels([LABEL[r].replace(" · ", "\n") for r in ALL], fontsize=7.5)
ax[1].set_ylabel("intrinsic dimension"); ax[1].set_title("(b) model-free intrinsic dimension", fontsize=9.5)
ax[1].legend(fontsize=8, title="estimator", title_fontsize=8)
for a in ax: a.grid(alpha=0.3); style_ax(a)
top_legend(fig, [mline(r) for r in ALL])
fig.suptitle("Fig 3 — geometry of the visited state (transformers probed at the last residual point)",
             y=1.10, fontsize=12)
fig.tight_layout(rect=[0, 0, 1, 0.93])
fig.savefig(f"{OUT}/fig3_geometry.png", dpi=130, bbox_inches="tight")
display(fig); plt.close(fig)
for r in ALL:
    sp = GEO[r]["spec"]; c = np.cumsum(sp)
    print(f"  {LABEL[r]:<34s} hull@90% {int(np.searchsorted(c,0.90))+1:>4d}  hull@99% {int(np.searchsorted(c,0.99))+1:>4d}"
          f"  TwoNN {GEO[r]['twonn']:.1f}  MLE {GEO[r]['mle']:.1f}")

---
## §4 — Editing the *readable* state

Write to the residual stream at residual point ℓ, at the current position, and roll out. The write
shapes the immediate prediction; that prediction then enters the observation buffer, and every later
step is recomputed from the buffer with no edit applied. **Any persistence therefore has to travel
through the observations** — which is precisely the property under test.

Two editors, both from the canonical suite: **readout injection** (linear-probe pseudoinverse — the
one that is inert on every architecture so far) and the **decoder-gradient oracle** (Adam on the
activation to match the true post-edit observation — the bracket that shows whether a
target-rendering activation exists at all).

In [ ]:
# [5] §4 setup — edit set, ground-truth zones, and the two activation editors.
N = N_EVAL
oe = edits.edit_object[:N].astype(int)
obs_e = edits.obs[:N].astype(np.float32)
gt_roll = edits.clean_obs[:N, ef:ef+K_ROLL, :].astype(np.float32)
tgt_pos = edits.positions[:N, ef, :N_OBJ, :].astype(np.float32)
pre_pos = edits.positions[:N, ef-1, :N_OBJ, :].astype(np.float32)
tgt4 = torch.from_numpy(tgt_pos.reshape(N, N_OBJ*2)).float().to(DEVICE)
ZONES = build_edit_zones(pre_pos=pre_pos, tgt_pos=tgt_pos, pre_vel=VEL_E[:N, ef-1],
                         edit_object=oe, sim=sim, n_obj=N_OBJ,
                         traj_pos=edits.positions[:N, ef:ef+K_ROLL, :N_OBJ, :].astype(np.float32),
                         gt_edited_traj=gt_roll)
tgt_obs_t = torch.from_numpy(gt_roll[:, 0]).float().to(DEVICE)
print(f"edit set: N={N}  ef={ef}  K={K_ROLL} | rays/sample: target {ZONES.target.sum(1).mean():.1f}, "
      f"ghost {ZONES.ghost.sum(1).mean():.1f}, differing {ZONES.differing.sum(1).mean():.1f}")

def fit_probe(X, Y):
    A = np.concatenate([X, np.ones((len(X),1),np.float32)], 1)
    sol, *_ = np.linalg.lstsq(A, Y, rcond=None)
    W = torch.tensor(sol[:-1], dtype=torch.float32, device=DEVICE)
    b = torch.tensor(sol[-1], dtype=torch.float32, device=DEVICE)
    Wp = torch.tensor(np.linalg.pinv(sol[:-1]), dtype=torch.float32, device=DEVICE)
    rmse = float(np.sqrt(((A @ sol - Y)**2).mean()))
    return W, b, Wp, rmse

@torch.no_grad()
def act_at(model, state, layer):
    model.state_view = "activations"; model.probe_layer = layer
    return model.flat_state(state)

def roll_act_edit(model, state, layer, h_new, steps=K_ROLL):
    """Rollout whose first step is produced under an activation edit (transformer),
    or an ordinary rollout from the written state (GRU — where the write is also the state)."""
    if hasattr(model, "rollout_with_edit"):
        with torch.no_grad():
            return model.rollout_with_edit(state, layer, h_new, steps).cpu().numpy()
    return roll(model, model.state_from_flat(h_new), steps)

def activation_editors(model, state, layer):
    """Readout injection + decoder-gradient oracle, both in activation space at `layer`."""
    Hs = states_at(model, layer if IS_TF[model_key(model)] else None)
    Tp = Hs.shape[1]
    W, b, Wp, prmse = fit_probe(Hs.reshape(-1, Hs.shape[-1]),
                                P_pos[:, :Tp].reshape(-1, N_OBJ*2))
    h0 = act_at(model, state, layer) if IS_TF[model_key(model)] else model.flat_state(state)
    out = {"Readout injection": h0 + (tgt4 - (h0 @ W + b)) @ Wp}
    h = h0.clone().requires_grad_(True)
    opt = torch.optim.Adam([h], lr=0.05)
    for _ in range(200):
        opt.zero_grad()
        if hasattr(model, "decode_with_edit"):
            pred = model.decode_with_edit(state, layer, h)
        else:
            pred = model.decode(model.state_from_flat(h))
        ((pred - tgt_obs_t)**2).mean().backward(); opt.step()
    out["Decoder gradient"] = h.detach()
    # Diagnostics that separate "the editor is broken" from "the editor works and the decoder
    # ignores it". Without these, an inert result is not interpretable.
    with torch.no_grad():
        hi = out["Readout injection"]
        d0 = model.decode_with_edit(state, layer, h0) if IS_TF[model_key(model)] \
             else model.decode(model.state_from_flat(h0))
        d1 = model.decode_with_edit(state, layer, hi) if IS_TF[model_key(model)] \
             else model.decode(model.state_from_flat(hi))
        diag = dict(
            probe_err_before=float((h0 @ W + b - tgt4).norm(dim=1).mean()),
            probe_err_after=float((hi @ W + b - tgt4).norm(dim=1).mean()),
            dh_rel=float(((hi - h0).norm(dim=1) / h0.norm(dim=1)).mean()),
            drender_rel=float(((d1 - d0).norm(dim=1) / d0.norm(dim=1)).mean()),
            probe_rmse=prmse)
    return h0, out, prmse, diag

_MODEL_KEY = {id(m): k for k, m in MODELS.items()}
def model_key(m): return _MODEL_KEY[id(m)]
print("editors ready")

In [ ]:
# [6] Fig 4 — editing the readable state, by residual point. Does a write to activations do anything?
ACT, ACT_STEP, DIAG, ACT_DH = {}, {}, {}, {}
for r in ALL:
    m = MODELS[r]; ACT[r] = {}; DIAG[r] = {}
    layers = LAYER_PTS if IS_TF[r] else [None]
    for L in layers:
        st = warm(m, obs_e, ef)
        h0, eds, prmse, diag = activation_editors(m, st, L)
        DIAG[r][L] = diag
        base = roll(m, st)
        card_u = edit_scorecard(base, ZONES, gt_roll)
        cards = {"Unsteered": card_u}
        ACT_STEP[(r, L, "Unsteered")] = edit_index_by_step(base, ZONES, gt_roll)
        for nm, hv in eds.items():
            re_ = roll_act_edit(m, st, L, hv)
            c = edit_scorecard(re_, ZONES, gt_roll)
            c["fidelity_ratio"] = fidelity_ratio(c, card_u); cards[nm] = c
            ACT_STEP[(r, L, nm)] = edit_index_by_step(re_, ZONES, gt_roll)
        cards["_probe_rmse"] = prmse
        ACT[r][L] = cards
        ACT_DH[(r, L)] = (eds["Decoder gradient"] - h0).detach()

LSTY = {0: dict(ls="-"), N_LAYERS//2: dict(ls="--"), N_LAYERS: dict(ls=":")}

def editor_figure(editor, fignum, subtitle, with_diag):
    """One figure per editor — Sevan 2026-08-04. The combined version hid the headline: the
    injection line sat exactly on top of the unsteered line and was invisible as a separate thing."""
    ncol = 3 if with_diag else 2
    fig, ax = plt.subplots(1, ncol, figsize=(6.0*ncol, 4.6))
    # (a) Edit Index at the edit frame, by residual point
    for r in TF_RUNS:
        ax[0].plot(LAYER_PTS, [ACT[r][L][editor]["edit_index"] for L in LAYER_PTS],
                   "-o", ms=5, color=COLOR[r])
        ax[0].axhline(ACT[r][N_LAYERS]["Unsteered"]["edit_index"], color=COLOR[r],
                      ls=(0, (1, 3)), lw=1.1, alpha=0.8)
    ax[0].axhline(ACT["H256"][None][editor]["edit_index"], color=COLOR["H256"], ls="--", lw=1.5)
    ax[0].set_xticks(LAYER_PTS)
    ax[0].set_xticklabels([LAYER_TICK.get(L, str(L)) for L in LAYER_PTS], fontsize=8)
    ax[0].set_xlabel("residual point edited"); ax[0].set_ylabel("Edit Index at the edit frame")
    ax[0].set_ylim(-1.08, 1.08)
    for y, lb in [(1.0, "= the edited world"), (0.0, "= equidistant"), (-1.0, "= the unedited world")]:
        ax[0].axhline(y, color="0.6", ls=":", lw=0.8)
        ax[0].annotate(lb, xy=(0.99, y), xycoords=("axes fraction", "data"), fontsize=7,
                       color="0.4", ha="right", va="bottom" if y < 1 else "top")
    ax[0].set_title("(a) does the write land? (step 0 only)", fontsize=9.5)
    # (b) Edit Index across the rollout, for three depths
    for r in TF_RUNS:
        for L in LAYER_TRIO:
            ys = ACT_STEP[(r, L, editor)]
            ax[1].plot(np.arange(len(ys)), ys, color=COLOR[r], lw=1.7, **LSTY[L])
    yu = ACT_STEP[("W16", N_LAYERS, "Unsteered")]
    ax[1].plot(np.arange(len(yu)), yu, color="#D55E00", lw=1.6)
    ax[1].set_ylim(-1.08, 1.08); ax[1].set_xlabel("rollout step (0 = the edit frame)")
    ax[1].set_ylabel("Edit Index")
    for y in (1.0, 0.0, -1.0): ax[1].axhline(y, color="0.6", ls=":", lw=0.8)
    ax[1].set_title("(b) does it hold, or revert? (free-run after the write)", fontsize=9.5)
    if with_diag:
        # (c) the write DID land — same units on both bars (dimensionless relative change)
        xs = np.arange(len(LAYER_PTS)); w = 0.38
        for r_i, r in enumerate(TF_RUNS):
            pass
        dh = [DIAG["W16"][L]["dh_rel"] for L in LAYER_PTS]
        dr = [DIAG["W16"][L]["drender_rel"] for L in LAYER_PTS]
        ax[2].bar(xs-w/2, dh, w, color="#0072B2", label="change in the state,  ‖Δh‖ / ‖h‖")
        ax[2].bar(xs+w/2, dr, w, color="#D55E00",
                  label="change in the render,  ‖Δdecode‖ / ‖decode‖")
        ax[2].set_xticks(xs)
        ax[2].set_xticklabels([LAYER_TICK.get(L, str(L)) for L in LAYER_PTS], fontsize=8)
        ax[2].set_xlabel("residual point edited")
        ax[2].set_ylabel("relative change (dimensionless)")
        ax[2].legend(fontsize=7.5, loc="upper right")
        ax[2].set_title("(c) the write is applied and lands exactly on the probe\n"
                        f"target (probe error {DIAG['W16'][N_LAYERS]['probe_err_after']:.1e} sim units) —\n"
                        "but the render barely moves · window 16", fontsize=8.5)
    for a_ in ax: a_.grid(alpha=0.3); style_ax(a_)
    handles = [mline(r) for r in ALL] + [
        Line2D([], [], color="0.35", **LSTY[L], lw=1.7,
               label=f"panel b: {LAYER_NAME.get(L, L)}") for L in LAYER_TRIO] + [
        Line2D([], [], color=COLOR["W16"], ls=(0, (1, 3)), lw=1.2, label="that model's unsteered level"),
        Line2D([], [], color="#D55E00", lw=1.6, label="unsteered (no edit)")]
    top_legend(fig, handles, ncol=4, y=1.0)
    fig.suptitle(f"Fig {fignum} — {subtitle}", y=1.13, fontsize=12)
    fig.tight_layout(rect=[0, 0, 1, 0.88])
    fig.savefig(f"{OUT}/fig{fignum}_{editor.split()[0].lower()}.png",
                dpi=130, bbox_inches="tight")
    display(fig); plt.close(fig)

editor_figure("Readout injection", "4a",
              "READOUT (pseudoinverse) INJECTION — the non-oracle editor. Colour = model.",
              with_diag=True)
editor_figure("Decoder gradient", "4b",
              "DECODER-GRADIENT ORACLE — for contrast. Colour = model.", with_diag=False)

rows = ["| model | residual point | probe error before → after (sim units) | ‖Δh‖/‖h‖ | ‖Δrender‖/‖render‖ "
        "| Edit Index: unsteered → injected |", "|---|---|---|---|---|---|"]
for r in ALL:
    for L in (LAYER_TRIO if IS_TF[r] else [None]):
        d = DIAG[r][L]
        rows.append(f"| {LABEL[r]} | {LAYER_NAME.get(L, 'single state') if IS_TF[r] else 'single state'} "
                    f"| {d['probe_err_before']:.2f} → {d['probe_err_after']:.1e} | {d['dh_rel']:.3f} "
                    f"| {d['drender_rel']:.4f} | {ACT[r][L]['Unsteered']['edit_index']:+.3f} → "
                    f"{ACT[r][L]['Readout injection']['edit_index']:+.3f} |")
display(Markdown(
    "**Table 3 — readout injection lands on the probe and not on the render.** The probe error going to "
    "~1e-6 proves the write is applied and that the linear probe reads exactly the requested position "
    "afterwards. `‖Δh‖/‖h‖` shows it is not a negligible perturbation of the state. `‖Δrender‖/‖render‖` "
    "and the Edit Index columns show what reaches the observation: essentially nothing. **This is the "
    "`readable ≠ grabbable` result, and it is a null result, not a broken editor.**\n\n" + "\n".join(rows)))

---
## §5 — Editing the *carried* state: the history-overwrite sweep

The decisive experiment. The only thing that persists is the observation buffer, so we overwrite its
last **n** frames with renders of the counterfactual world — the object travelling on a line that
arrives at the target — and sweep `n` from 0 to `state_span`.

This is the transformer's form of the **counterfactual state overwrite** that works best on the GRU,
and it answers: *how much history must be rewritten before an edit sticks?*

Two predictions were registered before running:
- **Sevan:** edits stick at ≲50% of the window overwritten.
- **Claude:** the required *absolute* number is roughly constant (~2–4 frames) regardless of window,
  because attention concentrates on recent frames — so the required *fraction* falls as the span grows.

They diverge most at `W16`. The adjudication is the **saturation point** — the smallest `n` reaching 90% of
that model's own maximum Edit Index — read in both currencies (absolute frames, and % of the carried span);
whichever is flat across windows is the real requirement. The weaker *crossover* threshold (Edit Index > 0)
is reported alongside, but a single frame can clear it, so it does not discriminate.

**Effective span.** At edit frame `ef = 20` a model has only ever seen 20 frames, so a run whose
`state_span` exceeds 20 has an *effective* carried state of 20 and the sweep tops out there. `W16`
(span 61) is therefore a full-context model in this setting.

**The two predictions are the endpoints of one scaling law.** Write the saturation point as
`n_sat ∝ span^β`. Then β = 0 is Claude's prediction (a fixed number of frames, independent of span)
and β = 1 is Sevan's (a fixed fraction of the state). Fitting β on the three runs is strictly more
informative than asking which of the two is "flatter", and it allows the answer to be *neither*.

In [ ]:
# [7] Fig 5 — how much history must be overwritten for the edit to stick?
REFL = np.array([sim["refl_min"], sim["refl_max"]], np.float32)
RAD = np.array([sim["radius"]]*N_OBJ, np.float32); COLc = np.tile(np.array([[1,1,1]],np.float32),(N_OBJ,1))
def _cfg(nf, noise):
    return SimConfig(seed=0, y_near=sim["y_near"], y_far=sim["y_far"], x_near=sim["x_near"],
                     x_far=sim["x_far"], n_objects=N_OBJ, radius=sim["radius"], n_frames=nf,
                     dt=sim["dt"], obs_res=sim["obs_res"], refl_min=sim["refl_min"],
                     refl_max=sim["refl_max"], fixed_reflectivities=True, obs_noise_std=noise,
                     boundary="open", always_in_frustum=False)
def render_traj(pos_seq, noise=0.0):
    _, _, it = render_scene(Scene(positions=pos_seq, velocities=np.zeros_like(pos_seq), radii=RAD,
                                 colors=COLc, reflectivities=REFL, config=_cfg(len(pos_seq), noise)))
    return it.astype(np.float32)

OBS_NOISE = float(sim["obs_noise_std"]); DT = float(sim["dt"])
MAX_SPAN = max(MODELS[r].state_span for r in TF_RUNS)
t_back = np.arange(MAX_SPAN)[::-1]                       # frames ef-1-… going back
cf_hist = np.zeros((N, MAX_SPAN, R), np.float32)
for i in range(N):
    o_, other = oe[i], 1 - oe[i]
    v = VEL_E[i, ef, o_]
    seq = np.zeros((MAX_SPAN, N_OBJ, 2), np.float32)
    seq[:, o_] = tgt_pos[i, o_][None] - v[None] * (t_back[:, None] + 1) * DT
    back = np.clip(ef - 1 - t_back, 0, None)
    seq[:, other] = edits.positions[i, back, other]
    cf_hist[i] = render_traj(seq, OBS_NOISE)

from pim.world_models.transformer import TransformerState

def overwrite_rollout(model, n_ow):
    """Replace the newest n_ow frames of the carried buffer with the counterfactual history.

    Built via `state_from_obs` so buffer padding and the `length` mask stay correct: only
    genuinely-seen frames are ever marked valid, so an overwrite can never smuggle in context
    the model did not have."""
    st = model.state_from_obs(torch.from_numpy(obs_e[:, :ef]).float().to(DEVICE))
    buf = st.obs_buffer.clone()
    if n_ow > 0:
        buf[:, -n_ow:] = torch.from_numpy(cf_hist[:, MAX_SPAN-n_ow:]).float().to(DEVICE)
    return roll(model, TransformerState(buf, st.length))

SWEEP = {}
for r in TF_RUNS:
    # the sweep tops out at the history that actually EXISTS: at frame ef the model has seen
    # ef frames, so a model whose span exceeds ef simply has a shorter effective carried state.
    span = min(MODELS[r].state_span, ef)
    ns = sorted(set([0,1,2,3,4,6,8,12,16,20] if span > 16 else list(range(span+1))))
    ns = [n for n in ns if n <= span]
    SWEEP[r] = {n: edit_scorecard(overwrite_rollout(MODELS[r], n), ZONES, gt_roll) for n in ns}
    print(f"  {LABEL[r]:<28s} span {span:>3}  swept n = {ns}")

EFF_SPAN = {r: min(MODELS[r].state_span, ef) for r in TF_RUNS}

def sweep_points(r):
    """Two thresholds per model, both read off that model's own curve:
      crossover  — smallest n with Edit Index > 0 (output stops being closer to the unedited world);
      saturation — smallest n reaching 90% of that model's own maximum Edit Index. This is the
                   adjudication metric: 'crossover' is a very low bar that n=1 can clear, whereas
                   saturation is where extra history stops buying anything."""
    ns = sorted(SWEEP[r]); ei = [SWEEP[r][n]["edit_index"] for n in ns]
    mx = max(ei)
    cross = next((n for n, v in zip(ns, ei) if v > 0), None)
    sat = next((n for n, v in zip(ns, ei) if v >= 0.9*mx), None)
    return ns, ei, mx, cross, sat

plt.style.use("default")
fig, ax = plt.subplots(1, 4, figsize=(22.5, 4.7), gridspec_kw={"width_ratios": [1, 1, 0.85, 1]})
for r in TF_RUNS:
    ns, ei, mx, cross, sat = sweep_points(r)
    sp = EFF_SPAN[r]
    ax[0].plot(ns, ei, "-o", ms=4.5, color=COLOR[r])
    ax[1].plot([100*n/sp for n in ns], ei, "-o", ms=4.5, color=COLOR[r])
    if sat is not None:                       # mark each curve's own saturation point
        ax[0].plot([sat], [SWEEP[r][sat]["edit_index"]], "*", ms=15, color=COLOR[r],
                   mec="0.2", mew=0.6, zorder=5)
        ax[1].plot([100*sat/sp], [SWEEP[r][sat]["edit_index"]], "*", ms=15, color=COLOR[r],
                   mec="0.2", mew=0.6, zorder=5)
for a_, xl, ttl in [
        (ax[0], "frames of history overwritten",
         "(a) absolute — does the threshold sit at the same frame count?"),
        (ax[1], "% of the carried state overwritten",
         "(b) relative — or at the same fraction of the state?")]:
    for y, lb in [(1.0, "= the edited world"), (0.0, "= equidistant"), (-1.0, "= the unedited world")]:
        a_.axhline(y, color="0.55", ls=":", lw=0.9)
        a_.annotate(lb, xy=(0.99, y), xycoords=("axes fraction", "data"), fontsize=7,
                    color="0.4", ha="right", va="bottom" if y < 1 else "top")
    a_.set_ylim(-1.12, 1.12); a_.set_xlabel(xl); a_.set_ylabel("Edit Index")
    a_.set_title(ttl, fontsize=9.5); a_.grid(alpha=0.3); style_ax(a_)

# (c) the adjudication panel. Frames and percent are different units, so plotting them on one raw
# axis would be meaningless; each series is instead divided by its OWN mean across windows. A series
# that is constant across windows sits flat on 1.0 — so "which bar group is flat" IS the answer.
xs = np.arange(len(TF_RUNS)); w = 0.38
sat_abs = np.array([sweep_points(r)[4] for r in TF_RUNS], float)
sat_pct = np.array([100*sweep_points(r)[4]/EFF_SPAN[r] for r in TF_RUNS], float)
for off, vals, hatch, lab in [(-w/2, sat_abs, None, "{:.0f} frames"),
                              (+w/2, sat_pct, "///", "{:.0f}% of span")]:
    norm = vals / max(vals.mean(), 1e-9)
    ax[2].bar(xs+off, norm, w, color=[COLOR[r] for r in TF_RUNS], edgecolor="0.25", lw=0.8,
              hatch=hatch, alpha=1.0 if hatch is None else 0.55)
    for x_, nv, rv in zip(xs+off, norm, vals):
        ax[2].annotate(lab.format(rv), xy=(x_, nv), xytext=(0, 4), textcoords="offset points",
                       ha="center", va="bottom", fontsize=7.5)
ax[2].axhline(1.0, color="0.3", ls="--", lw=1.2)
_sp = np.array([EFF_SPAN[r] for r in TF_RUNS], float)
_beta = float(np.polyfit(np.log(_sp), np.log(sat_abs), 1)[0])
ax[2].annotate(f"saturation ≈ span$^{{{_beta:.2f}}}$\n(0 = fixed frames, 1 = fixed fraction)",
               xy=(0.5, 0.98), xycoords="axes fraction", fontsize=8.5, color="0.2",
               ha="center", va="top",
               bbox=dict(boxstyle="round,pad=0.35", fc="#fdf6e3", ec="0.7", lw=0.7))
ax[2].set_ylim(0, 2.15)
ax[2].set_xticks(xs); ax[2].set_xticklabels([f"window {MODELS[r].cfg.window}\n(span {EFF_SPAN[r]})"
                                             for r in TF_RUNS], fontsize=8)
ax[2].set_ylabel("saturation point ÷ its own mean across windows")
ax[2].set_xlabel("(% uses the effective span — the history available at the edit frame)", fontsize=7.5)
ax[2].set_title("(c) solid = absolute frames · hatched = % of span\n"
                "neither is flat — the exponent below says how it really scales", fontsize=9)
ax[2].grid(alpha=0.3, axis="y"); style_ax(ax[2])

# (d) the notebook's thesis, made visible: a write to the READABLE state is transient by
# construction (the next step recomputes it from the buffer), while overwriting the CARRIED state
# persists. Both are shown against the same unsteered floor, on the same bounded scale.
STEP_CURVES = []
for r in TF_RUNS:
    sat = sweep_points(r)[4]
    STEP_CURVES.append((f"history overwrite, n = {sat} ({LABEL[r]})", COLOR[r], "-",
                        edit_index_by_step(overwrite_rollout(MODELS[r], sat), ZONES, gt_roll)))
st_d = warm(MODELS["W16"], obs_e, ef)
_, eds_d, _, _ = activation_editors(MODELS["W16"], st_d, N_LAYERS)
STEP_CURVES.append(("activation edit at the last residual point (window 16)", COLOR["W16"], ":",
                    edit_index_by_step(roll_act_edit(MODELS["W16"], st_d, N_LAYERS,
                                                     eds_d["Decoder gradient"]), ZONES, gt_roll)))
STEP_CURVES.append(("unsteered (no edit), window 16", "#D55E00", "-",
                    edit_index_by_step(roll(MODELS["W16"], st_d), ZONES, gt_roll)))
for lab, c_, ls_, ys in STEP_CURVES:
    ax[3].plot(np.arange(len(ys)), ys, ls=ls_, color=c_, lw=1.8,
               marker="o" if ls_ == "-" else "s", ms=3.5,
               mfc="none" if ls_ == ":" else c_)
for y, lb in [(1.0, "= the edited world"), (0.0, "= equidistant"), (-1.0, "= the unedited world")]:
    ax[3].axhline(y, color="0.55", ls=":", lw=0.9)
    ax[3].annotate(lb, xy=(0.99, y), xycoords=("axes fraction", "data"), fontsize=7,
                   color="0.4", ha="right", va="bottom" if y < 1 else "top")
ax[3].set_ylim(-1.12, 1.12); ax[3].set_xlabel("rollout step (0 = the edit frame)")
ax[3].set_ylabel("Edit Index")
ax[3].set_title("(d) does it hold? solid = carried-state overwrite\ndotted = readable-state write",
                fontsize=9)
ax[3].grid(alpha=0.3); style_ax(ax[3])

handles = [Patch(facecolor=COLOR[r], label=LABEL[r]) for r in TF_RUNS] + [
    Line2D([], [], ls="none", marker="*", ms=13, color="0.35",
           label="saturation point (90% of that model's own maximum Edit Index)"),
    Line2D([], [], color="0.35", ls="-", lw=1.8, label="history overwrite (carried state) — panel d"),
    Line2D([], [], color="0.35", ls=":", lw=1.8, label="activation edit (readable state) — panel d"),
    Line2D([], [], color="#D55E00", lw=1.8, label="unsteered (no edit)")]
top_legend(fig, handles, ncol=4, y=1.02)
fig.suptitle("Fig 5 — how much history must be overwritten before the edit sticks?  Registered predictions: "
             "Sevan — a fixed fraction of the state (panel b flat); "
             "Claude — a fixed number of frames (panel a flat).", y=1.13, fontsize=11.5)
fig.tight_layout(rect=[0, 0, 1, 0.87])
fig.savefig(f"{OUT}/fig5_history_overwrite.png", dpi=130, bbox_inches="tight")
display(fig); plt.close(fig)

rows = ["| model | architectural span | effective span (history available at ef) | crossover n | "
        "**saturation n** | % of effective span | % of architectural span | max Edit Index |",
        "|---|---|---|---|---|---|---|---|"]
for r in TF_RUNS:
    ns, ei, mx, cross, sat = sweep_points(r); sp = EFF_SPAN[r]; arch = MODELS[r].state_span
    rows.append(f"| {LABEL[r]} | {arch} | {sp} | {cross if cross is not None else '—'} | **{sat}** | "
                f"{100*sat/sp:.0f}% | {100*sat/arch:.0f}% | {mx:+.2f} |")
display(Markdown(
    "**Table 2 — the decisive numbers.** *Crossover* = smallest number of overwritten frames at which the "
    "output stops being closer to the unedited world (Edit Index > 0); a single frame can clear it, so it does "
    "not discriminate. *Saturation* = smallest n reaching 90% of that model's own maximum Edit Index — the "
    "point past which more history buys nothing. This is the number that adjudicates the two registered "
    "predictions: if **saturation n** is roughly constant across windows the requirement is a fixed frame "
    "count (Claude); if a **percentage column** is roughly constant it is a fixed fraction (Sevan).\n\n"
    "> **Which denominator?** At edit frame ef the model has only ever seen `ef` frames, so a run whose "
    "architectural `state_span` exceeds `ef` has a shorter *effective* carried state. The two percentage "
    "columns differ only for such a run (here: window 16, architectural 61 vs effective 20) — read the "
    "**effective** column, since it is the history that actually exists, but both are shown so the "
    "difference is not hidden.\n\n" + "\n".join(rows)))

---
## §5b — Does the edit *stick*? Edit Index across the rollout as more history is overwritten

*Added 2026-08-04 at Sevan's request.* §5 answered "how much history" with a single number per model
(the saturation point, measured at the edit frame). This asks the sharper question: for each amount of
overwritten history `n`, what does the Edit Index do **across the free-running rollout** — does it hold,
or revert toward the unedited world?

Reading it: a curve that starts high and falls back toward the unsteered floor is an edit that did **not**
take; a curve that stays flat and high is one that did. The `n` at which the curves stop falling is the
honest answer to "how much history must be rewritten for the edit to stick", and it need not equal the
step-0 saturation point of §5.

In [ ]:
# [8] Fig 7 — Edit Index across the rollout, swept over how many past observations are overwritten.
STEP_BY_N = {r: {n: edit_index_by_step(overwrite_rollout(MODELS[r], n), ZONES, gt_roll)
                 for n in sorted(SWEEP[r])} for r in TF_RUNS}

plt.style.use("default")
fig, ax = plt.subplots(1, len(TF_RUNS), figsize=(6.0*len(TF_RUNS), 4.7), squeeze=False)
ax = ax[0]
from matplotlib.colors import LinearSegmentedColormap
cmap = LinearSegmentedColormap.from_list("nsweep", ["#c9d6e3", "#4a90c4", "#0b3d66"])
for i, r in enumerate(TF_RUNS):
    ns = sorted(STEP_BY_N[r]); sat = sweep_points(r)[4]
    nmax = max(ns)
    for n in ns:
        ys = STEP_BY_N[r][n]
        is_sat = (n == sat)
        ax[i].plot(np.arange(len(ys)), ys, color=cmap(n/max(nmax, 1)),
                   lw=2.6 if is_sat else 1.3, zorder=4 if is_sat else 2,
                   marker="o" if is_sat else None, ms=3.5)
    yu = STEP_BY_N[r][0]
    ax[i].plot(np.arange(len(yu)), yu, color="#D55E00", lw=2.0, zorder=5)
    ax[i].annotate(f"n = {sat} (saturation)", xy=(len(STEP_BY_N[r][sat])-1, STEP_BY_N[r][sat][-1]),
                   xytext=(-6, 10), textcoords="offset points", fontsize=7.5,
                   color=cmap(sat/max(nmax, 1)), ha="right")
    for y, lb in [(1.0, "= the edited world"), (0.0, "= equidistant"), (-1.0, "= the unedited world")]:
        ax[i].axhline(y, color="0.6", ls=":", lw=0.8)
        ax[i].annotate(lb, xy=(0.99, y), xycoords=("axes fraction", "data"), fontsize=7,
                       color="0.4", ha="right", va="bottom" if y < 1 else "top")
    ax[i].set_ylim(-1.1, 1.1); ax[i].set_xlabel("rollout step (0 = the edit frame)")
    ax[i].set_ylabel("Edit Index" if i == 0 else "")
    ax[i].set_title(f"{LABEL[r]} · effective span {EFF_SPAN[r]}", fontsize=9.5)
    ax[i].grid(alpha=0.3); style_ax(ax[i])
sm = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(0, max(max(SWEEP[r]) for r in TF_RUNS)))
cb = fig.colorbar(sm, ax=ax.tolist(), fraction=0.016, pad=0.012)
cb.set_label("n = past observations overwritten", fontsize=9)
top_legend(fig, [Line2D([], [], color="#D55E00", lw=2.0, label="n = 0 (unsteered — no edit at all)"),
                 Line2D([], [], color="0.35", lw=2.6, marker="o", ms=4,
                        label="that model's saturation point (bold)")], ncol=2, y=1.0)
fig.suptitle("Fig 7 — does the edit stick? Edit Index over the free-running rollout, "
             "swept over how many past observations were overwritten", y=1.07, fontsize=12)
fig.savefig(f"{OUT}/fig7_stick_by_n.png", dpi=130, bbox_inches="tight")
display(fig); plt.close(fig)

rows = ["| model | n overwritten | Edit Index @ step 0 | @ step 7 | @ step 14 | retained (step 14 / step 0) |",
        "|---|---|---|---|---|---|"]
for r in TF_RUNS:
    for n in sorted(STEP_BY_N[r]):
        y = STEP_BY_N[r][n]
        keep = "—" if y[0] <= 0 else f"{y[-1]/y[0]:.2f}"
        rows.append(f"| {LABEL[r]} | {n} | {y[0]:+.2f} | {y[len(y)//2]:+.2f} | {y[-1]:+.2f} | {keep} |")
display(Markdown("**Table 4 — does the edit hold?** *Retained* is the step-14 Edit Index divided by the "
                 "step-0 value, defined only where the edit landed (step 0 > 0). A value near 1 means the "
                 "edit persisted through the free run; near 0 means it decayed back toward "
                 "ambiguity.\n\n" + "\n".join(rows)))

In [ ]:
# [9] Fig 6 — observation waterfalls (canonical spec: gray on dark, noisy context above a dashed
#     edit line, each column its OWN free-run from step 0, green target / red-dash ghost, top legend).
N_CTX = 6
DARK, TXT, TICK, EDIT_C = "#0a0a14", "#a3adc2", "#808a9d", "#fa8850"
TGT_C, GHO_C = "#00E676", "#FF5252"
ctx = edits.obs[:N, ef-N_CTX:ef, :].astype(np.float32)
def _cx(mk):
    i = np.where(mk)[0]; return i.mean() if i.size else np.nan
tcx = np.array([_cx(ZONES.target[i]) for i in range(N)])
gcx = np.array([_cx(ZONES.ghost[i]) for i in range(N)])
SAMP = list(np.argsort(ZONES.teleport * (ZONES.ghost.sum(1) >= 3))[::-1][:3])

def waterfall(col_titles, bodies, samples, suptitle, fname):
    nc = len(col_titles)
    fig, axes = plt.subplots(len(samples), nc, figsize=(2.9*nc, 3.3*len(samples)),
                             squeeze=False, facecolor=DARK)
    for r_, smp in enumerate(samples):
        for c_ in range(nc):
            a_ = axes[r_][c_]; a_.set_facecolor(DARK)
            panel = np.clip(np.concatenate([ctx[smp], bodies[c_][smp]], 0), 0, 1)
            a_.imshow(panel, aspect="auto", origin="upper", cmap="gray", vmin=0, vmax=1,
                      interpolation="nearest")
            for sp in a_.spines.values(): sp.set_edgecolor(TICK)
            a_.axhline(N_CTX-0.5, color=EDIT_C, lw=1.4, ls="--", alpha=0.95)
            if not np.isnan(tcx[smp]): a_.axvline(tcx[smp], color=TGT_C, lw=1.6, alpha=0.9)
            if not np.isnan(gcx[smp]): a_.axvline(gcx[smp], color=GHO_C, ls="--", lw=1.6, alpha=0.9)
            if r_ == 0: a_.set_title(col_titles[c_], fontsize=8, color=TXT)
            if c_ == 0:
                a_.set_ylabel(f"sample {smp}\n(teleport {ZONES.teleport[smp]:.1f})\nsim frame",
                              fontsize=8, color=TXT)
                a_.set_yticks([0, N_CTX, N_CTX+7, N_CTX+14])
                a_.set_yticklabels([ef-N_CTX, ef, ef+7, ef+14], fontsize=7)
            else: a_.set_yticks([])
            if r_ == len(samples)-1: a_.set_xlabel("ray", fontsize=8, color=TXT)
            else: a_.set_xticklabels([])
            a_.tick_params(colors=TICK, labelsize=7)
    hs = [Line2D([0],[0], color=TGT_C, lw=2.2, label="object target location"),
          Line2D([0],[0], color=GHO_C, ls="--", lw=2.2, label="ghost (pre-edit) location"),
          Line2D([0],[0], color=EDIT_C, ls="--", lw=2.2,
                 label=f"edit applied here ({N_CTX} noisy context frames above; every row below is that "
                       f"column's OWN free-run, step 0 = frame {ef})")]
    fig.legend(handles=hs, loc="upper center", ncol=2, fontsize=8.5, frameon=False,
               labelcolor=TXT, bbox_to_anchor=(0.5, 0.955))
    fig.suptitle(suptitle, y=1.0, fontsize=10.5, color=TXT)
    fig.tight_layout(rect=[0, 0, 1, 0.90])
    fig.savefig(f"{OUT}/{fname}", dpi=130, bbox_inches="tight", facecolor=DARK)
    display(fig); plt.close(fig); print("saved", fname)

REF = "W16"; span_ref = min(MODELS[REF].state_span, ef); L_MID = N_LAYERS // 2
n_show = [n for n in (1, 4, 16) if n <= span_ref]
st_ref = warm(MODELS[REF], obs_e, ef)
# Both activation editors, at BOTH the middle and the last residual point. The middle point is where
# the injection perturbs the render most (Table 3: 0.036 vs 0.015), so this is its most favourable
# showing, not a strawman. Column titles name the EDITOR, never just the site — the earlier
# "activation edit (last residual point)" label did not say which editor produced it.
_, eds_last, _, _ = activation_editors(MODELS[REF], st_ref, N_LAYERS)
_, eds_mid,  _, _ = activation_editors(MODELS[REF], st_ref, L_MID)

def _ei(body):
    return edit_scorecard(body, ZONES, gt_roll)["edit_index"]

spec = [("GT (sim)", gt_roll, False),
        ("unsteered\n(no edit)", roll(MODELS[REF], st_ref), True),
        (f"pseudoinverse injection\n(residual point {L_MID}, middle)",
         roll_act_edit(MODELS[REF], st_ref, L_MID, eds_mid["Readout injection"]), True),
        (f"pseudoinverse injection\n(residual point {N_LAYERS}, last)",
         roll_act_edit(MODELS[REF], st_ref, N_LAYERS, eds_last["Readout injection"]), True),
        (f"decoder-gradient oracle\n(residual point {N_LAYERS}, last)",
         roll_act_edit(MODELS[REF], st_ref, N_LAYERS, eds_last["Decoder gradient"]), True)]
spec += [(f"history overwrite\nn = {n}", overwrite_rollout(MODELS[REF], n), True) for n in n_show]
spec += [(f"full overwrite\nn = {span_ref}", overwrite_rollout(MODELS[REF], span_ref), True)]
cols = [t + (f"\nEdit Index {_ei(b):+.2f}" if sc else "\n(the target)") for t, b, sc in spec]
bodies = [b for _, b, _ in spec]
waterfall(cols, bodies, SAMP,
          f"Fig 6 — {LABEL[REF]}: writing to the readable state (residual stream) versus "
          f"overwriting the carried state (observation buffer). Three largest-teleport samples.",
          "fig6_waterfalls.png")

### More waterfalls — unselected samples

*Added 2026-08-04 at Sevan's request.* Fig 6 shows the three **largest-teleport** samples, which is a
selected view. Fig 8 shows the first six samples of the edits split with **no selection at all**, so the
typical case is visible alongside the favourable one.

**On predictability of the post-edit trajectory.** In this world `direction_noise_std = speed_noise_std = 0`,
so **velocity is exactly constant** — verified on the edits split, `max |v[t+1] − v[t]| = 0`. The only
stochasticity is `position_noise_std = 0.04` per step, applied to the position as a random walk (the velocity
is untouched). Over the 15-step rollout the ballistic displacement averages **1.09 units** while the
accumulated noise is **0.15 units** (1σ), so ~88% of the post-edit motion is determined by
`(position, velocity)`. A model that recovers those two numbers is *expected* to track the ground truth —
including a late "drift", which is ballistic motion, not noise. See §5c.

In [ ]:
# [10] Fig 8 — the same comparison on six UNSELECTED samples (the first six of the edits split).
SAMP2 = list(range(6))
n_show2 = [n for n in (4, 16) if n <= span_ref]
spec2 = [("GT (sim)", gt_roll, False),
         ("unsteered\n(no edit)", roll(MODELS[REF], st_ref), True),
         (f"pseudoinverse injection\n(residual point {L_MID}, middle)",
          roll_act_edit(MODELS[REF], st_ref, L_MID, eds_mid["Readout injection"]), True),
         (f"decoder-gradient oracle\n(residual point {N_LAYERS}, last)",
          roll_act_edit(MODELS[REF], st_ref, N_LAYERS, eds_last["Decoder gradient"]), True)]
spec2 += [(f"history overwrite\nn = {n}", overwrite_rollout(MODELS[REF], n), True) for n in n_show2]
spec2 += [(f"full overwrite\nn = {span_ref}", overwrite_rollout(MODELS[REF], span_ref), True)]
cols2 = [t + (f"\nEdit Index {_ei(b):+.2f}" if sc else "\n(the target)") for t, b, sc in spec2]
bodies2 = [b for _, b, _ in spec2]
waterfall(cols2, bodies2, SAMP2,
          f"Fig 8 — {LABEL[REF]}: the same editor line-up on six UNSELECTED samples "
          f"(edits split, samples 0-5)",
          "fig8_waterfalls_unselected.png")

### §5c — Is the post-edit trajectory actually predictable, or is the model guessing?

A rollout that tracks the ground truth *including its late drift* looks suspicious. It is not. The check
below decomposes the ground-truth motion into the part that is a deterministic function of
`(position, velocity)` — which the counterfactual history explicitly supplies — and the part that is
genuinely unpredictable process noise, and compares the model's rollout against a **pure ballistic
extrapolation** that has no learning in it at all.

In [ ]:
# [11] Predictability audit: how much of the post-edit motion is deterministic, and can a
#      zero-learning ballistic extrapolator reproduce the "drift"?
tb_pos = edits.positions[:N, ef:ef+K_ROLL, :N_OBJ, :].astype(np.float32)
v_ef   = VEL_E[:N, ef]                                             # exactly constant by construction
ball   = tb_pos[:, :1] + v_ef[:, None] * np.arange(K_ROLL)[None, :, None, None] * DT
resid  = np.linalg.norm(tb_pos - ball, axis=-1)                    # (N, K, n_obj) sim units
moved  = np.linalg.norm(tb_pos[:, -1] - tb_pos[:, 0], axis=-1)
print(f"velocity constancy on the edits split: max |v[t+1]-v[t]| = "
      f"{np.abs(VEL_E[:N, 1:] - VEL_E[:N, :-1]).max():.2e}  (0 => exactly ballistic)")
print(f"ground-truth displacement over {K_ROLL-1} steps : {moved.mean():.3f} units (mean over objects)")
print(f"deviation from pure ballistic at the last step  : {resid[:, -1].mean():.3f} units "
      f"(= accumulated position noise, sigma={sim['position_noise_std']}/step)")
print(f"=> {100*(1 - resid[:, -1].mean()/moved.mean()):.0f}% of the post-edit motion is fixed by "
      "(position, velocity) alone; the rest is unpredictable in principle.")
srows = ["| sample | teleport (units) | |v| (units/frame) | GT displacement over rollout | deviation "
         "from ballistic at last step |", "|---|---|---|---|---|"]
for smp in SAMP2:
    o_ = oe[smp]
    srows.append(f"| {smp} | {ZONES.teleport[smp]:.2f} | {np.linalg.norm(v_ef[smp, o_]):.4f} | "
                 f"{moved[smp, o_]:.3f} | {resid[smp, -1, o_]:.3f} |")
display(Markdown("**Table 5 — the 'drift' is ballistic, not a lucky guess.** The counterfactual history "
                 "injected by the history-overwrite editor is a straight line travelling at the true "
                 "velocity, so the model can read that velocity off the frames it is given and extrapolate. "
                 "The residual column is the only part it could not have known.\n\n" + "\n".join(srows)))

---
## §6 — Why is the injection inert? The decoder's own gradient versus the probe's row space

*Added 2026-08-04 at Sevan's request.* §4 established that pseudoinverse injection does not move the
render. This section asks **why**, and the question has a sharp geometric form.

Readout injection can only ever move `h` inside the probe's **row space**: it produces
`Δh_pinv = (target − (A h + b)) A⁺`, which lies in `row(A)` **by construction**. Meanwhile the decoder has
an opinion about which direction in `h` would actually change the render toward the target — the descent
direction `−∇_h ‖decode(h) − obs_target‖²`, evaluated at the *unedited* activation. If those two directions
are close, injection should work. If the decoder's direction lies mostly outside the probe's row space,
injection **cannot** work however the target is chosen, and the §4 null is a statement about geometry
rather than about tuning.

### Definitions

| term | formula | units | notes |
|---|---|---|---|
| **decoder-descent direction** `g` | `−∇_h ‖decode(h) − gt_obs[ef]‖²` at `h = h0` (unedited) | direction in `h` | the first step the decoder-gradient oracle would take. Sign flipped so that a positive cosine means "the decoder wants to move the way injection moves". |
| **pseudoinverse direction** `Δh_pinv` | `(target − (A h0 + b)) A⁺` | direction in `h` | exactly what readout injection applies; lies in `row(A)` by construction |
| **converged oracle displacement** `Δh_dg` | `h_final − h0` after 200 Adam steps | direction in `h` | where the decoder actually ends up, not just its first step |
| **cosine** | `⟨u,v⟩ / (‖u‖‖v‖)`, **per sample, then averaged** | −1…+1 | never computed on averaged vectors. Reported with the **angle** in degrees, because cos 0.9 is a 26° angle, not "90% similar". |
| **row-space fraction** | `‖Qᵀg‖ / ‖g‖`, `Q` = orthonormal basis of `row(A)` | 0…1 | the share of the decoder's direction that injection can reach at all — a **hard ceiling** on the editor |
| **chance level** | `√(d/H)`, `d = 4` probe outputs, `H = 256` | 0…1 | a random direction already has this much of its norm in any 4-dimensional subspace: **0.125**. Always report the ratio to chance, never the raw fraction alone. |
| **shuffled control** | the same cosine, with `Δh_pinv` taken from a *different* sample | −1…+1 | the empirical null. The mean cosine of random pairs is **0**, not `1/√H` — `1/√H` is the per-pair standard deviation. |

In [ ]:
# [12] Fig 9 — is the decoder's preferred direction inside the probe's row space?
def decoder_descent(model, state, layer, target_obs):
    """g = -grad_h ||decode(h) - target||^2 at the UNEDITED activation h0, per sample."""
    h0 = act_at(model, state, layer) if IS_TF[model_key(model)] else model.flat_state(state)
    h = h0.clone().requires_grad_(True)
    pred = (model.decode_with_edit(state, layer, h) if hasattr(model, "decode_with_edit")
            else model.decode(model.state_from_flat(h)))
    g, = torch.autograd.grad(((pred - target_obs)**2).sum(), h)
    return h0.detach(), (-g).detach()

def cos_rows(U, V):
    """Per-sample cosine between matching rows of U and V (never on averaged vectors)."""
    u = U / U.norm(dim=1, keepdim=True).clamp_min(1e-12)
    v = V / V.norm(dim=1, keepdim=True).clamp_min(1e-12)
    return (u * v).sum(1)

GEOM = {}
rng_sh = np.random.default_rng(0)
for r in ALL:
    m = MODELS[r]; GEOM[r] = {}
    for L in (LAYER_PTS if IS_TF[r] else [None]):
        Hs = states_at(m, L if IS_TF[r] else None); Tp = Hs.shape[1]
        W, b, Wp, _ = fit_probe(Hs.reshape(-1, Hs.shape[-1]), P_pos[:, :Tp].reshape(-1, N_OBJ*2))
        st = warm(m, obs_e, ef)
        h0, g = decoder_descent(m, st, L, tgt_obs_t)
        dpinv = (tgt4 - (h0 @ W + b)) @ Wp              # lies in row(A) by construction
        dconv = ACT_DH[(r, L)]                          # converged oracle displacement
        Q = torch.linalg.qr(W)[0]                       # (H, d) orthonormal basis of row(A)
        frac = (Q.T @ g.T).norm(dim=0) / g.norm(dim=1).clamp_min(1e-12)
        perm = rng_sh.permutation(len(g))
        GEOM[r][L] = dict(cos_grad=cos_rows(g, dpinv).cpu().numpy(),
                          cos_conv=cos_rows(dconv, dpinv).cpu().numpy(),
                          cos_shuf=cos_rows(g, dpinv[perm]).cpu().numpy(),
                          rowfrac=frac.cpu().numpy(),
                          d=int(W.shape[1]), H=int(W.shape[0]))
CHANCE = np.sqrt(GEOM["W16"][N_LAYERS]["d"] / GEOM["W16"][N_LAYERS]["H"])

plt.style.use("default")
fig, ax = plt.subplots(1, 2, figsize=(13.5, 4.7))
CS = {"cos_grad": dict(ls="-", marker="o", ms=4.5), "cos_conv": dict(ls="--", marker="^", ms=5)}
for r in TF_RUNS:
    for key, stl in CS.items():
        mu = [GEOM[r][L][key].mean() for L in LAYER_PTS]
        sd = [GEOM[r][L][key].std() for L in LAYER_PTS]
        ax[0].errorbar(LAYER_PTS, mu, yerr=sd, color=COLOR[r], lw=1.6, capsize=3, **stl)
for key, stl in CS.items():
    ax[0].axhline(GEOM["H256"][None][key].mean(), color=COLOR["H256"], ls=stl["ls"], lw=1.5)
shuf = np.concatenate([GEOM[r][L]["cos_shuf"] for r in TF_RUNS for L in LAYER_PTS])
ax[0].axhspan(shuf.mean()-shuf.std(), shuf.mean()+shuf.std(), color="0.75", alpha=0.45, zorder=0)
ax[0].annotate(f"shuffled-pair control: {shuf.mean():+.3f} ± {shuf.std():.3f} (1 sd)",
               xy=(0.03, shuf.mean()+shuf.std()), xycoords=("axes fraction", "data"), fontsize=7.5,
               color="0.3", va="bottom")
ax[0].axhline(0, color="0.5", ls=":", lw=0.9)
ax[0].set_xticks(LAYER_PTS); ax[0].set_xticklabels([LAYER_TICK.get(L, str(L)) for L in LAYER_PTS], fontsize=8)
ax[0].set_xlabel("residual point"); ax[0].set_ylabel("cosine with the pseudoinverse direction")
ax[0].set_title("(a) does the decoder want to move the way injection moves?\nerror bars = 1 sd across samples",
                fontsize=9.5)
sec = ax[0].secondary_yaxis("right", functions=(lambda c: np.degrees(np.arccos(np.clip(c, -1, 1))),
                                                lambda a: np.cos(np.radians(a))))
sec.set_ylabel("angle (degrees)", fontsize=9)
for r in TF_RUNS:
    mu = [GEOM[r][L]["rowfrac"].mean() for L in LAYER_PTS]
    sd = [GEOM[r][L]["rowfrac"].std() for L in LAYER_PTS]
    ax[1].errorbar(LAYER_PTS, mu, yerr=sd, color=COLOR[r], lw=1.6, marker="o", ms=4.5, capsize=3)
ax[1].axhline(GEOM["H256"][None]["rowfrac"].mean(), color=COLOR["H256"], ls="--", lw=1.5)
ax[1].axhline(CHANCE, color="#D55E00", ls=":", lw=1.6)
ax[1].annotate(f"chance for a random direction: √(d/H) = √(4/256) = {CHANCE:.3f}",
               xy=(0.03, CHANCE), xycoords=("axes fraction", "data"), fontsize=7.5,
               color="#D55E00", va="bottom")
ax[1].set_ylim(0, None)
ax[1].set_xticks(LAYER_PTS); ax[1].set_xticklabels([LAYER_TICK.get(L, str(L)) for L in LAYER_PTS], fontsize=8)
ax[1].set_xlabel("residual point")
ax[1].set_ylabel("fraction of the decoder direction inside row(A)")
ax[1].set_title("(b) how much of what the decoder wants can injection reach at all?\nthis is the editor's hard ceiling",
                fontsize=9.5)
for a_ in ax: a_.grid(alpha=0.3); style_ax(a_)
handles = [mline(r) for r in ALL] + [
    Line2D([], [], color="0.35", lw=1.6, label="panel a: first decoder-descent step", **CS["cos_grad"]),
    Line2D([], [], color="0.35", lw=1.6, label="panel a: converged oracle displacement", **CS["cos_conv"])]
top_legend(fig, handles, ncol=3, y=1.0)
fig.suptitle(f"Fig 9 — why readout injection is inert: the decoder's preferred direction is nearly "
             f"orthogonal to the subspace injection can write in  (N = {N} samples, per-sample then averaged)",
             y=1.12, fontsize=11.5)
fig.tight_layout(rect=[0, 0, 1, 0.90])
fig.savefig(f"{OUT}/fig9_gradient_vs_rowspace.png", dpi=130, bbox_inches="tight")
display(fig); plt.close(fig)

rows = ["| model | residual point | cosine(decoder descent, pseudoinverse dir) | angle | shuffled control "
        "| converged oracle vs pseudoinverse dir | row-space fraction | ÷ chance |",
        "|---|---|---|---|---|---|---|---|"]
for r in ALL:
    for L in (LAYER_TRIO if IS_TF[r] else [None]):
        d_ = GEOM[r][L]
        cg, cv, cs_, rf = d_["cos_grad"], d_["cos_conv"], d_["cos_shuf"], d_["rowfrac"]
        rows.append(f"| {LABEL[r]} | {LAYER_NAME.get(L, 'single state') if IS_TF[r] else 'single state'} "
                    f"| {cg.mean():+.3f} ± {cg.std():.3f} "
                    f"| {np.degrees(np.arccos(np.clip(cg.mean(), -1, 1))):.1f}° "
                    f"| {cs_.mean():+.3f} ± {cs_.std():.3f} | {cv.mean():+.3f} ± {cv.std():.3f} "
                    f"| {rf.mean():.3f} ± {rf.std():.3f} | {rf.mean()/CHANCE:.2f}× |")
display(Markdown(
    "**Table 6 — the injection's hard ceiling.** All cosines are computed **per sample and then averaged**. "
    "The *angle* column translates the mean cosine: two equal-length vectors θ apart differ by "
    "`2·sin(θ/2)` of their length, so ~87° means the two directions are almost unrelated. The "
    "*shuffled control* pairs each decoder direction with a **different** sample's pseudoinverse direction "
    "and is the empirical null (its mean is 0, not `1/√H`). The *row-space fraction* is the share of "
    "the decoder's preferred direction that injection can reach **at all**; chance for a random direction "
    f"is `√(d/H) = √(4/256) = {CHANCE:.3f}`, so the final column is the enrichment over chance."
    "\n\n" + "\n".join(rows)))

---
## §7 — Summary

In [ ]:
# [13] Computed summary.
print("=========== Summary — every number below is computed here, not asserted ===========")
print(f"\n1. Quality gate — next-step RMSE vs clean (GRU {PRED['H256']['next']:.4f}, "
      f"noise floor {bl.noise_floor_rmse:.4f}):")
for r in TF_RUNS:
    print(f"     {LABEL[r]:<28s} {PRED[r]['next']:.4f}  ({PRED[r]['next']/PRED['H256']['next']:.2f}x GRU)")
gate = all(PRED[r]["next"] <= 1.15 * PRED["H256"]["next"] for r in TF_RUNS)
print(f"   => {'Passed — the editability numbers below are interpretable' if gate else 'Caution — at least one transformer is a materially worse predictor; gate the §4 claims on it'}")

print("\n2. Where is the state readable? Position R² (linear probe) by residual point:")
for r in TF_RUNS:
    vals = " ".join(f"{L}:{DEPTH[r][L]['pos_r2_linear']:.2f}" for L in LAYER_PTS)
    best = max(LAYER_PTS, key=lambda L: DEPTH[r][L]["pos_r2_linear"])
    print(f"     {LABEL[r]:<28s} {vals}   peak at point {best}  (GRU {GRU_REC['pos_r2_linear']:.2f})")

print("\n3. Editing the readable state (activation write) — Edit Index by residual point:")
for r in TF_RUNS:
    u = ACT[r][N_LAYERS]["Unsteered"]["edit_index"]
    ro = " ".join(f"{L}:{ACT[r][L]['Readout injection']['edit_index']:+.2f}" for L in LAYER_PTS)
    dg = " ".join(f"{L}:{ACT[r][L]['Decoder gradient']['edit_index']:+.2f}" for L in LAYER_PTS)
    print(f"     {LABEL[r]:<28s} unsteered {u:+.2f}")
    print(f"       readout injection  {ro}")
    print(f"       decoder gradient   {dg}")
print(f"     {LABEL['H256']:<28s} readout {ACT['H256'][None]['Readout injection']['edit_index']:+.2f} | "
      f"decoder gradient {ACT['H256'][None]['Decoder gradient']['edit_index']:+.2f} | "
      f"unsteered {ACT['H256'][None]['Unsteered']['edit_index']:+.2f}")

print("\n4. Editing the carried state — history-overwrite sweep:")
for r in TF_RUNS:
    ns, ei, mx, cross, sat = sweep_points(r); sp = EFF_SPAN[r]
    print(f"     {LABEL[r]:<28s} span {sp:>3} | crossover n={cross} | saturation n={sat} "
          f"({100*sat/sp:.0f}% of span) | max Edit Index {mx:+.2f}")

print("\n   Adjudicating the registered predictions:")
sat_a = np.array([sweep_points(r)[4] for r in TF_RUNS], float)
sat_p = np.array([100*sweep_points(r)[4]/EFF_SPAN[r] for r in TF_RUNS], float)
sat_arch = np.array([100*sweep_points(r)[4]/MODELS[r].state_span for r in TF_RUNS], float)
print(f"     effective span         : {[int(EFF_SPAN[r]) for r in TF_RUNS]}")
print(f"     absolute frames        : {[int(v) for v in sat_a]}")
print(f"     % of effective span    : {[round(v) for v in sat_p]}")
print(f"     % of architectural span: {[round(v) for v in sat_arch]}   (differs only where state_span > ef)")
# The two predictions are the two ENDPOINTS of one scaling law, saturation ~ span**beta:
#   beta = 0 -> a fixed frame count (Claude);  beta = 1 -> a fixed fraction (Sevan).
# Fitting beta is strictly more informative than asking which of the two is "flatter".
spans = np.array([EFF_SPAN[r] for r in TF_RUNS], float)
BETA = float(np.polyfit(np.log(spans), np.log(sat_a), 1)[0]) if len(spans) >= 2 else float("nan")
print(f"\n     scaling exponent beta in  saturation ~ span^beta  :  beta = {BETA:.2f}")
print("       beta = 0 would be Claude's prediction (a fixed number of frames)")
print("       beta = 1 would be Sevan's prediction (a fixed fraction of the state)")
if BETA < 0.25:
    verdict = "Claude's prediction is supported — the requirement is close to a fixed frame count."
elif BETA > 0.75:
    verdict = "Sevan's prediction is supported — the requirement is close to a fixed fraction of the state."
else:
    verdict = (f"NEITHER prediction is right, and the truth sits between them: beta = {BETA:.2f}, i.e. the "
               f"requirement grows roughly like the SQUARE ROOT of the available history. Overwriting "
               f"{sat_a[0]:.0f} of {spans[0]:.0f} frames ({sat_p[0]:.0f}%) suffices at the short window, but "
               f"only {sat_a[-1]:.0f} of {spans[-1]:.0f} ({sat_p[-1]:.0f}%) at the long one — the absolute "
               "count does grow with the span (against Claude), but far more slowly than proportionally "
               "(against Sevan).")
print(f"     => {verdict}")
print(f"     (fit on {len(spans)} points — thin; treat beta as an order-of-magnitude statement, and note "
      "the two shortest spans nearly tie in absolute frames.)")
act_curve = [c for lab, _, _, c in STEP_CURVES if lab.startswith("activation edit")][0]
ow_curves = [c for lab, _, _, c in STEP_CURVES if lab.startswith("history overwrite")]
uns_curve = [c for lab, _, _, c in STEP_CURVES if lab.startswith("unsteered")][0]
print("\n5. Transient vs persistent (Edit Index at step 0 -> last step):")
print(f"     activation edit, readable state   {act_curve[0]:+.2f} -> {act_curve[-1]:+.2f}")
for (lab, _, _, c_) in [x for x in STEP_CURVES if x[0].startswith("history overwrite")]:
    print(f"     {lab:<48s} {c_[0]:+.2f} -> {c_[-1]:+.2f}")
print(f"     unsteered (no edit)               {uns_curve[0]:+.2f} -> {uns_curve[-1]:+.2f}")
print("   => a high step-0 Edit Index is not evidence of a durable edit; read the last step too.")
print("\n   fidelity ratio (GT-traj RMSE of the editor / of unsteered; > 1 = the edit made the rollout "
      "worse than doing nothing) at the last residual point:")
for r in TF_RUNS:
    fr_ = {nm: ACT[r][N_LAYERS][nm]["fidelity_ratio"] for nm in ("Readout injection", "Decoder gradient")}
    print(f"     {LABEL[r]:<28s} " + "  ".join(f"{k} {v:.2f}" for k, v in fr_.items()))
print("\nSaved figures:", sorted(os.listdir(OUT)))

---
## §4d — Is it uneditable for editors *other* than Pseudoinverse Injection?

§4 tested two activation editors (**Pseudoinverse Injection** and **Decoder Grad Steering k=1**). A single
failing editor never shows that a state is uneditable — it shows that *one* write mechanism failed. This
section adds three more, all at the **last residual point** (the decoder's input, where the readable state
lives and a write has its best chance), for every transformer run plus the GRU reference.

**Editors added** (canonical names, `../METRICS_AND_EDITORS.md`): **Global PCA Projection (PI)** ·
**MLP Grad Steering** · **Iterative Nullspace Projection @29 (R² corrected)**.

**Deliberately skipped, and why** — the ask was for speed, and two of these are not merely slow but
ill-posed here:
- **Local PCA Geodesic @120** — 120 per-sample local-PCA refits × 5 models. Slow, and nothing about it is
  transformer-specific; the GRU result already stands in for it.
- **Multistep Steering (PI) @16** — requires the edit to *survive* into the next step so the next nudge builds
  on it. On a transformer an activation edit **cannot** survive: the next step recomputes the residual stream
  from the observation buffer. The mechanism is undefined here rather than untested.
- **Decoder Grad Steering k=15** — same reason. Optimising `h` against a 15-step rollout is meaningless when
  the write only affects step 0.

> **Read step 0, not persistence.** An activation edit is **one-step by construction** on a transformer — the
> residual stream is recomputed from the observation buffer every step, so decay here is *architecture*, not the
> GRU's reversion failure. A one-step effect is the ceiling. Persistence lives in the **history overwrite**
> editor (§5), which writes the carried state instead.

> **Read the `probe error after` column carefully — it is only a fair diagnostic for one of the three.**
> It measures the **linear** readout's distance to the target after the write. That is exactly what
> **Global PCA Projection (PI)** is trying to minimise, so for that arm it is the right check. It is *not*
> for the other two: **MLP Grad Steering** targets a **different (MLP) probe**, and
> **Iterative Nullspace Projection** deliberately **shrinks** each probe's target by its own R²
> (`target_k = μ + R²_k(target − μ)`), so it is *designed* not to drive the linear readout to the target.
> A large value in those two rows is expected and is not evidence that the write failed.

### Current results (updated 2026-08-06)

**The transformer is still uneditable — now against five activation editors, not one.** Every added editor
leaves all three transformer runs on the unedited side, and none of them degrades the model
(**fidelity 0.99–1.00** throughout, so these are genuine inert results rather than the Edit Index drifting
toward 0 through damage):

| editor | W2 | W4 | W16 | GRU H256 |
|---|---|---|---|---|
| Unsteered | −0.66 | −0.67 | −0.68 | −0.67 |
| Global PCA Projection (PI) | −0.50 | −0.52 | −0.53 | −0.52 |
| MLP Grad Steering | −0.60 | −0.64 | −0.64 | −0.59 |
| Iterative Nullspace Projection @29 (R² corrected) | **−0.47** | **−0.39** | **−0.48** | −0.40 |

Best is INLP at **−0.39…−0.48** against an unsteered floor of −0.66…−0.68 — the same ordering, and nearly the
same values, as on the GRU. **Window size is irrelevant**: W2, W4 and W16 agree to within 0.09 on every
editor, so nothing here depends on how much history the transformer can see.

**The one genuinely transformer-specific finding is upstream of editability: the write itself lands less
well.** On the GRU, Global PCA Projection drives the linear readout error from 3.23 to **0.062** — it
essentially achieves its objective, and the model then ignores it. On the transformers the *same* editor only
reaches **1.98–2.20** from 3.22, despite a comparable ‖Δ‖/‖h‖ (0.083–0.088). So on the transformer these
editors are not cleanly in the "succeeded on its own terms and was ignored" regime that makes the GRU result
sharp — part of the failure is that the activation at the last residual point resists being written to at all
while staying on the state manifold. That is a weaker, more confounded negative than the GRU's, and worth
saying plainly rather than reporting the Edit Index alone.

**Caveats.** One residual point (the last) rather than the full layer sweep — chosen because that is where the
readable state lives and where a write has its best chance, so it is the favourable case, not a random one.
Step 0 only, which is the architectural ceiling for an activation edit (§5's history overwrite is the channel
that persists). Local PCA Geodesic was skipped for cost; Multistep Steering and Decoder Grad k=15 are
ill-posed on a transformer activation edit, as noted above.

In [ ]:
# [14] §4d — three more activation editors at the last residual point. Speed-first: no per-sample loops.
from pim.editors import fit_state_subspace, manifold_steer
from pim.extractors import MLPExtractor, StateDefinition
from dataclasses import replace as _replace
import time as _time

EXTRA = ["Global PCA Projection (PI)", "MLP Grad Steering",
         "Iterative Nullspace Projection @29 (R² corrected)"]
EXT, EXT_DIAG = {}, {}
_t0 = _time.perf_counter()

for r in ALL:
    m = MODELS[r]; L = N_LAYERS if IS_TF[r] else None
    st = warm(m, obs_e, ef)
    Hs = states_at(m, L if IS_TF[r] else None)                 # (N, T, D) activation bank
    D = Hs.shape[-1]; Tp = Hs.shape[1]
    Xb = Hs.reshape(-1, D).astype(np.float64)
    Yb = P_pos[:, :Tp].reshape(-1, N_OBJ * 2).astype(np.float64)
    W, b, Wp, prmse = fit_probe(Xb.astype(np.float32), Yb.astype(np.float32))
    h0 = act_at(m, st, L) if IS_TF[r] else m.flat_state(st)
    inject = lambda h, t: h + (t - (h @ W + b)) @ Wp

    eds = {}
    # 1 — Global PCA Projection (PI): alternating inject <-> project onto the 90%-variance subspace
    sub = fit_state_subspace(Hs, var_threshold=0.90)
    sub = _replace(sub, mean=sub.mean.to(DEVICE), basis=sub.basis.to(DEVICE),
                   explained_variance_ratio=sub.explained_variance_ratio.to(DEVICE))
    eds["Global PCA Projection (PI)"] = manifold_steer(h0, tgt4, inject, sub, n_iters=50)

    # 2 — MLP Grad Steering: frozen 1x128 STEERING probe (deliberately NOT the readability standard),
    #     then batched Adam on the activation through it.
    sdef = StateDefinition(name="pos", state_shape=(N_OBJ * 2,), extract_fn=lambda x: x)
    mlp = MLPExtractor(D, sdef, mlp_hidden=128, n_epochs=12, lr=5e-3)
    mlp.fit(Hs, P_pos[:, :Tp].astype(np.float32), device=DEVICE)
    mlp = mlp.to(DEVICE).eval()
    # batched Adam through the frozen probe. `pim.editors.gradient_steer` is single-sample
    # (it reshapes the prediction to batch 1), and a 192-sample python loop x5 models is exactly
    # the kind of cost this section is avoiding. Same objective, one optimiser over the batch.
    _h = h0.clone().detach().requires_grad_(True)
    _opt = torch.optim.Adam([_h], lr=0.01)
    for _ in range(200):
        _opt.zero_grad()
        ((mlp(_h).reshape(_h.shape[0], -1) - tgt4) ** 2).mean().backward()
        _opt.step()
    eds["MLP Grad Steering"] = _h.detach()

    # 3 — Iterative Nullspace Projection @29 (R² corrected): 29 orthogonal probe subspaces, injected
    #     at once with each target shrunk by that probe's own R².
    Xr = Xb.copy(); casc = []
    ntr = int(0.8 * len(Xr))
    for _ in range(29):
        aug = np.concatenate([Xr[:ntr], np.ones((ntr, 1))], 1)
        sol, *_ = np.linalg.lstsq(aug, Yb[:ntr], rcond=None)
        pr = Xr[ntr:] @ sol[:-1] + sol[-1]
        r2k = float(1 - ((pr - Yb[ntr:]) ** 2).sum() / ((Yb[ntr:] - Yb[:ntr].mean(0)) ** 2).sum())
        Ak = sol[:-1].T
        _, s_, vt = np.linalg.svd(Ak, full_matrices=False)
        Bk = vt[: int((s_ > s_[0] * 1e-8).sum())].T
        casc.append(dict(A=Ak, b=sol[-1], r2=r2k, pinv=np.linalg.pinv(Ak)))
        Xr = Xr - (Xr @ Bk) @ Bk.T
    mu = Yb[:ntr].mean(0)
    h0n = h0.detach().cpu().numpy().astype(np.float64)
    tgtn = tgt4.detach().cpu().numpy().astype(np.float64)
    dh = np.zeros_like(h0n)
    for c_ in casc:
        dh += ((mu + c_["r2"] * (tgtn - mu)) - (h0n @ c_["A"].T + c_["b"])) @ c_["pinv"].T
    eds["Iterative Nullspace Projection @29 (R² corrected)"] = torch.from_numpy(
        (h0n + dh).astype(np.float32)).to(DEVICE)

    base = roll(m, st); card_u = edit_scorecard(base, ZONES, gt_roll)
    card_u["fidelity_ratio"] = 1.0                      # it IS the reference
    EXT[r] = {"Unsteered": card_u}
    for nm, hv in eds.items():
        rl = roll_act_edit(m, st, L, hv)
        c = edit_scorecard(rl, ZONES, gt_roll)
        c["fidelity_ratio"] = fidelity_ratio(c, card_u)
        with torch.no_grad():
            d0 = m.decode_with_edit(st, L, h0) if IS_TF[r] else m.decode(m.state_from_flat(h0))
            d1 = m.decode_with_edit(st, L, hv) if IS_TF[r] else m.decode(m.state_from_flat(hv))
            c["probe_err_after"] = float((hv @ W + b - tgt4).norm(dim=1).mean())
            c["dh_rel"] = float(((hv - h0).norm(dim=1) / h0.norm(dim=1)).mean())
            c["drender_rel"] = float(((d1 - d0).norm(dim=1) / d0.norm(dim=1)).mean())
        EXT[r][nm] = c
    EXT[r]["_probe_err_before"] = float((h0 @ W + b - tgt4).norm(dim=1).mean())
    EXT[r]["_cascade_r2"] = (casc[0]["r2"], casc[-1]["r2"])
    print(f"{r}: probe RMSE {prmse:.3f} | cascade R² {casc[0]['r2']:.2f}->{casc[-1]['r2']:.2f}")
print(f"\nthree extra editors x {len(ALL)} models in {_time.perf_counter()-_t0:.0f}s")

rows = ["| model | editor | Edit Index (step 0) ↑ | Target RMSE ↓ | Ghost RMSE ↓ | fidelity ↓ | probe error after ↓ | ‖Δ‖/‖h‖ | render change |",
        "|---|---|---|---|---|---|---|---|---|"]
for r in ALL:
    for nm in ["Unsteered"] + EXTRA:
        c = EXT[r][nm]
        extra = (f"| {c['probe_err_after']:.3f} | {c['dh_rel']:.3f} | {c['drender_rel']:.3f} |"
                 if nm != "Unsteered" else f"| {EXT[r]['_probe_err_before']:.3f} | — | — |")
        rows.append(f"| {r} | {nm} | **{c['edit_index']:+.2f}** | {c['target_rmse']:.3f} | "
                    f"{c['ghost_rmse']:.3f} | {c['fidelity_ratio']:.2f} " + extra)
display(Markdown(
    "**Table — three more activation editors, at the last residual point.** Read the Edit Index against each "
    "model's own `Unsteered` row. The last three columns are the diagnostics that separate *'the editor is "
    "broken'* from *'the editor worked and the model ignored it'*: **probe error after** is how close the write "
    "got the readout to the target (near 0 = the write did what it was asked), **‖Δ‖/‖h‖** is how far the "
    "activation moved, and **render change** is how much the decoded observation moved as a result. A write "
    "with probe error ≈ 0, a real ‖Δ‖, and a near-zero render change is an editor that succeeded on its own "
    "terms and was ignored downstream.\n\n" + "\n".join(rows)))